# Train LeRobot ACT Policies on Google Colab (GPU)

Trains one **ACT (Action Chunking with Transformers)** policy per manipulation skill, using **LeRobot 0.6.1**.

This version takes **one combined zip** containing every skill's dataset as a top-level folder
(`open_drawer/`, `pick_plate/`, `pick_mug/`, `pick_bottle/`, `place_plate/`, `pour_water/`, each a real
LeRobot v3.0 `LeRobotDataset` produced by `scripts/build_clean_dataset.py`). It auto-discovers whichever
skill folders are actually inside the zip and trains a separate policy for each - no per-skill upload,
no manual list of filenames to edit.

### Step 1: Set Runtime to GPU
Go to **Runtime -> Change runtime type -> T4 GPU (or any GPU) -> Save**.

### Step 2: Upload the combined dataset zip
Drag `data/butler_demos_all.zip` from your local repo into the Colab file explorer on the left
(or upload to Drive and mount it, then point `ZIP_PATH` below at that location).

In [ ]:
# 0. Configuration

ZIP_PATH = "/content/butler_demos_all.zip"  # where you uploaded the combined dataset zip
EXTRACT_ROOT = "/content/dataset"

DEFAULT_STEPS = 10000
DEFAULT_BATCH_SIZE = 8

# Optional: exclude specific skills even if their folder is present in the zip
# (e.g. to re-run just one skill without retraining everything).
SKIP_SKILLS = set()  # e.g. {"open_drawer"} to skip a skill you already trained

# Optional: override steps/batch_size for specific skills instead of the defaults above.
PER_SKILL_OVERRIDES = {
    # "pour_water": {"steps": 15000, "batch_size": 8},
}

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install LeRobot with training extras
!pip install --upgrade pip
!pip install "lerobot[training]>=0.6.1"

In [ ]:
# 3. Extract the combined zip once, then auto-discover which skill folders it contains.
# A folder counts as a skill dataset only if it has meta/info.json - anything else in the
# zip (stray files, __MACOSX/, etc.) is ignored rather than guessed at.
import os

assert os.path.exists(ZIP_PATH), f"{ZIP_PATH} not found - upload data/butler_demos_all.zip first."
!unzip -o -q "{ZIP_PATH}" -d "{EXTRACT_ROOT}"

dataset_roots = {}
for name in sorted(os.listdir(EXTRACT_ROOT)):
    candidate = os.path.join(EXTRACT_ROOT, name)
    if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "meta", "info.json")):
        dataset_roots[name] = candidate

assert dataset_roots, f"No skill datasets found under {EXTRACT_ROOT} - check the zip contents."
print(f"Discovered {len(dataset_roots)} skill dataset(s): {list(dataset_roots)}")
for skill, root in dataset_roots.items():
    print(f"  [{skill}] -> {root}  (meta files: {os.listdir(os.path.join(root, 'meta'))})")

In [ ]:
# 4. Apply SKIP_SKILLS and build the final per-skill training plan.

skills_to_train = {
    skill: root for skill, root in dataset_roots.items() if skill not in SKIP_SKILLS
}
assert skills_to_train, "Every discovered skill is in SKIP_SKILLS - nothing left to train."

plan = {}
for skill in skills_to_train:
    override = PER_SKILL_OVERRIDES.get(skill, {})
    plan[skill] = {
        "root": skills_to_train[skill],
        "steps": override.get("steps", DEFAULT_STEPS),
        "batch_size": override.get("batch_size", DEFAULT_BATCH_SIZE),
    }

print("Training plan:")
for skill, cfg in plan.items():
    print(f"  [{skill}] steps={cfg['steps']} batch_size={cfg['batch_size']} root={cfg['root']}")

In [ ]:
# 5. (Recommended) Validate each dataset against the Butler schema before spending
# GPU time on it - catches a malformed export early rather than mid-training.
import subprocess

!git clone --depth 1 https://github.com/MUHAMMAD-AZEEM-AZAM/Butler-AI.git /content/repo
%cd /content/repo

for skill, cfg in plan.items():
    print(f"\n=== validating {skill} ===")
    result = subprocess.run(
        ["python", "-m", "stage3_policy.learned.validate_dataset", "--root", cfg["root"], "--min-successful-episodes", "1"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"WARNING: {skill} dataset did not fully validate - inspect the report above before trusting the checkpoint.")

In [ ]:
# 6. Launch ACT training on GPU, one policy per skill in the plan.
# Sequential, not parallel - each run gets the full GPU, and a failure in one
# skill's training does not stop the others from being attempted.

for skill, cfg in plan.items():
    output_dir = f"/content/outputs/train/act_{skill}"
    print(f"\n{'='*70}\nTraining '{skill}' -> {output_dir}\n{'='*70}")
    !lerobot-train \
      --dataset.repo_id=local/butler_demos_{skill} \
      --dataset.root={cfg['root']} \
      --policy.type=act \
      --policy.device=cuda \
      --policy.push_to_hub=false \
      --output_dir={output_dir} \
      --job_name=act_{skill} \
      --steps={cfg['steps']} \
      --batch_size={cfg['batch_size']} \
      --save_freq=2500 \
      --seed=1000 \
      --wandb.enable=false

In [ ]:
# 7. Package and download every trained checkpoint that actually completed.
import os
from google.colab import files

for skill in plan:
    ckpt_dir = f"/content/outputs/train/act_{skill}/checkpoints/last/pretrained_model"
    if not os.path.isdir(ckpt_dir):
        print(f"[{skill}] no checkpoint found at {ckpt_dir} - training likely did not complete, skipping download.")
        continue
    zip_name = f"/content/trained_act_{skill}_checkpoint.zip"
    !zip -r {zip_name} {ckpt_dir}
    files.download(zip_name)
    print(f"[{skill}] checkpoint download initiated: {zip_name}")